In [1]:
import numpy as np
import pandas as pd
from data_split import podzialy_walidacyjne, ZIARNA, krawedzie
from ocena import etykieta_liczbowa, f_opis_i_y, prog_decyzyjny, ocena
from modele import model_A, model_B, model_C, model_D


In [2]:
NUMERY = [0, 1, 2, 3, 4]
MODELE = [model_A, model_B, model_C, model_D]
wyniki = []
oceny_par = []

In [3]:
for ziarno in ZIARNA:
    for numer in NUMERY:
        p = podzialy_walidacyjne[(ziarno, numer)]
        y_trening = etykieta_liczbowa(p["trening"])
        y_walidacja = etykieta_liczbowa(p["walidacja"])
        opis_t, y_t, idx = f_opis_i_y(ziarno, numer) # opis i etykiety dla zbioru testowego

        for model in MODELE:
            if model == model_C:
                ziarno_modelu = ziarno*10 + numer
            else:
                ziarno_modelu = ziarno
            
            wyniki_modelu = model(p, y_trening, ziarno_modelu)
            
            prog = prog_decyzyjny(y_walidacja, wyniki_modelu["walidacja"]) # wybor progu            
            
            tabela = ocena(y_t, wyniki_modelu["test"], opis_t, prog) # ocena na teście
            tabela["model"] = model.__name__
            tabela["ziarno"] = ziarno
            tabela["falda"] = numer
            wyniki.append(tabela)

            pary = pd.DataFrame({
                "model": model.__name__,
                "ziarno": ziarno,
                "falda": numer,
                "wiersz": idx,
                "nadawca": krawedzie.loc[idx, "nadawca"].to_numpy(),
                "adresat": krawedzie.loc[idx, "odbiorca"].to_numpy(),
                "y": y_t,
                "ocena": wyniki_modelu["test"],
                "prog": prog,
            })
            oceny_par.append(pary)

In [4]:
wyniki = pd.concat(wyniki, ignore_index=True)
wyniki = wyniki[["model", "ziarno", "falda", "podgrupa", "miara", "wartosc"]]
wyniki.to_csv("wyniki_etap3.csv", index=False)
print("wyniki shape", wyniki.shape) # 25 * 4* 29 linijek

wyniki shape (2900, 6)


In [5]:
oceny_par = pd.concat(oceny_par, ignore_index=True)
oceny_par.to_csv("oceny_par_etap3.csv", index=False)
print("oceny_par shape", oceny_par.shape) # 371 *4 *25

oceny_par shape (37100, 9)


In [6]:
caly = wyniki[wyniki.podgrupa == "caly_test"]
zestawienie = caly.groupby(["model", "miara"])["wartosc"].agg(["mean", "std"])
print(zestawienie)
zestawienie.to_csv("zestawienie_caly_test.csv")   

                                 mean       std
model   miara                                  
model_A auc_pr               0.264689  0.002041
        auc_w_parach_klubow  0.500000  0.000000
        f1                   0.000000  0.000000
        f1_makro             0.423734  0.000678
        roc_auc              0.500000  0.000000
model_B auc_pr               0.928202  0.012747
        auc_w_parach_klubow  0.500000  0.000000
        f1                   0.902136  0.014154
        f1_makro             0.933773  0.009690
        roc_auc              0.957958  0.006788
model_C auc_pr               0.930410  0.021481
        auc_w_parach_klubow  0.721245  0.154329
        f1                   0.892931  0.023508
        f1_makro             0.927113  0.016080
        roc_auc              0.971504  0.008125
model_D auc_pr               0.901888  0.021211
        auc_w_parach_klubow  0.830966  0.102429
        f1                   0.807122  0.033172
        f1_makro             0.869631  0

Mieszanie etykiet


In [7]:
MODELE_MIESZANIE = [model_B, model_C, model_D]
wyniki_mieszanie = []

In [8]:
for ziarno in ZIARNA:
    for numer in NUMERY:
        p = podzialy_walidacyjne[(ziarno, numer)]
        y_trening = etykieta_liczbowa(p["trening"])
        y_walidacja = etykieta_liczbowa(p["walidacja"])
        opis_t, y_t, idx = f_opis_i_y(ziarno, numer)

        y_trening_mieszane = np.random.default_rng(ziarno * 67 + numer).permutation(y_trening)

        for model in MODELE_MIESZANIE:
            if model == model_C:
                ziarno_modelu = ziarno*10 + numer
            else:
                ziarno_modelu = ziarno
            
            wyniki_modelu = model(p, y_trening_mieszane, ziarno_modelu)
            
            prog = prog_decyzyjny(y_walidacja, wyniki_modelu["walidacja"]) # wybor progu            
            
            tabela = ocena(y_t, wyniki_modelu["test"], opis_t, prog) # ocena na teście
            tabela["model"] = model.__name__
            tabela["ziarno"] = ziarno
            tabela["falda"] = numer
            wyniki_mieszanie.append(tabela)
        

In [ ]:
wyniki_mieszanie = pd.concat(wyniki_mieszanie, ignore_index=True)
wyniki_mieszanie = wyniki_mieszanie[["model", "ziarno", "falda", "podgrupa", "miara", "wartosc"]]
wyniki_mieszanie.to_csv("wyniki_mieszanie_etap3.csv", index=False)
print("wyniki mieszanie shape", wyniki_mieszanie.shape) # 25 * 3* 29 linijek

wyniki mieszanie shape (2175, 6)


In [11]:
caly_m = wyniki_mieszanie[wyniki_mieszanie.podgrupa == "caly_test"]
grupy_m = caly_m.groupby(["model", "miara"])["wartosc"]
zestawienie_mieszanie = pd.DataFrame({"srednia": grupy_m.mean(), "odchylenie": grupy_m.std()})
zestawienie_mieszanie.to_csv("zestawienie_mieszanie.csv")